# Train a regression model

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

## Load the Data

In [ ]:
epe_filename = 'epedataset.csv'
df = pd.read_csv(epe_filename)

In [ ]:
df.head()

In [ ]:
df.drop(['Unnamed: 0'], axis=1, inplace=True)

In [ ]:
y_scaling = 1000000.0

In [ ]:
X = df.iloc[:,:-11]
y = df.iloc[:,-11:]*y_scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01) 

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
batch_size = 512

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
# Swish activation function
class Swish(nn.Module):
    def forward(self, input):
        return input * torch.sigmoid(input)
    
# Feed-forward network
class FeedForwardNetwork(nn.Module):
    def __init__(self, input_size, num_hidden_layers, neurons_per_layer, output_size):
        super(FeedForwardNetwork, self).__init__()

        # Construct layers
        layers = []

        # Input layer
        layers.append(nn.Linear(input_size, neurons_per_layer))
        layers.append(Swish())

        # Hidden layers
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(neurons_per_layer, neurons_per_layer))
            layers.append(Swish())

        # Output layer
        layers.append(nn.Linear(neurons_per_layer, output_size))

        # Combine layers
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
X_train.shape

In [ ]:
model = FeedForwardNetwork(X_train.shape[1], 4, 400, 11).to(device)

## Build and Train the model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = F.mse_loss(outputs.squeeze(), batch_y)  # Assuming MSE is the desired error metric
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train error: {train_loss:.4f}, Test error: {test_loss:.4f}")

    return train_errors, test_errors

In [ ]:
epochs = 200
lr = 0.001

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
model_out = train_model(model, data_loader, test_data_loader, loss_fn, optimizer, epochs)

In [ ]:
def plot_loss(train_errors, test_errors, filename):
    plt.figure(figsize=(10, 6))
    plt.plot(train_errors, color='black', linestyle='-', label='Train Loss')
    plt.plot(test_errors, color='black', linestyle='--', label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.legend()
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
filename = 'epelosses.png'

In [ ]:
train_loss, test_loss = model_out

In [ ]:
plot_loss(train_loss, test_loss, filename)

In [ ]:
model.eval()

batch = next(iter(test_data_loader))
inputs, labels = batch
with torch.no_grad():
    outputs = model(inputs)

In [ ]:
outputs.shape

In [ ]:
n = 6  # number of rows
m = 3  # number of columns
fig, axes = plt.subplots(n, m, figsize=(m * 5, n * 4))  # Adjust figure size as needed

for i in range(n):
    for j in range(m):
        ax = axes[i, j]
        index = i * m + j  # Calculate the index for the subplot

        if index < len(outputs):
            ax.plot(outputs.cpu().numpy()[index], color='black', label='Model EPE', linestyle='--')
            ax.plot(labels.cpu().numpy()[index], color='black', label='True EPE', linestyle='-')
            ax.set_title(f'Subplot {index + 1}')
            ax.legend()
        else:
            ax.axis('off')  # Turn off axis for empty subplots

# Adjust the layout
plt.tight_layout()
plt.savefig('epecomparison.png', dpi=300)
plt.show()